In [1]:
import pandas as pd

customers_df = pd.read_json("noahs-customers.jsonl", lines=True)
orders_df = pd.read_json("noahs-orders.jsonl", convert_dates=['ordered', 'shipped'], lines=True)
products_df = pd.read_json("noahs-products.jsonl", lines=True)

SKU IDs:

- `PET` pets
- `COL` collectible
- `DLI` deli
- `BKY` bakery
- `HOM` household goods

In [2]:
norm_orders_df = orders_df.explode('items', ignore_index=True)
norm_orders_df = pd.concat([norm_orders_df.drop(columns=['items']), pd.json_normalize(norm_orders_df['items'])], axis=1)

In [3]:
df = norm_orders_df.join(
    customers_df.set_index('customerid'),
    on='customerid',
).join(products_df.set_index('sku'), on='sku')


Find Neighbor's order

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 426541 entries, 0 to 426540
Data columns (total 19 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   orderid         426541 non-null  int64         
 1   customerid      426541 non-null  int64         
 2   ordered         426541 non-null  datetime64[us]
 3   shipped         426541 non-null  datetime64[us]
 4   total           426541 non-null  float64       
 5   sku             426541 non-null  str           
 6   qty             426541 non-null  int64         
 7   unit_price      426541 non-null  float64       
 8   name            426541 non-null  str           
 9   address         426541 non-null  str           
 10  citystatezip    426541 non-null  str           
 11  birthdate       426541 non-null  str           
 12  phone           426541 non-null  str           
 13  timezone        426541 non-null  str           
 14  lat             426541 non-null  float64       

In [27]:
order_date = df[df['customerid'] == 2550]['ordered'].item()
df[df['customerid'] == 2550]

,orderid,customerid,ordered,shipped,total,sku,qty,unit_price,name,address,citystatezip,birthdate,phone,timezone,lat,long,desc,wholesale_cost,dims_cm
34013,18146,2550,2017-07-22 11:27:48,2017-07-22 11:27:48,3.08,HOM2761,2,1.54,Robert Morton,145-51 107th Ave,"Jamaica, NY 11435",1999-07-08,917-288-9635,America/New_York,40.689595,-73.804871,Rug Cleaner,1.43,"[19.6, 11.7, 0.2]"


In [37]:
import datetime
def valid_order_dates(s: pd.Series):
    return (datetime.time(4, 0, 0) <= s.dt.time) & (s.dt.time <= datetime.time(5, 0, 0)) & (order_date <= s)

In [39]:
df[df['sku'].str.contains('BKY') & valid_order_dates(df['ordered']) & (df['ordered'] == df['shipped'])].sort_values('ordered')

,orderid,customerid,ordered,shipped,total,sku,qty,unit_price,name,address,citystatezip,birthdate,phone,timezone,lat,long,desc,wholesale_cost,dims_cm
87522,45075,2749,2018-04-20 04:59:06,2018-04-20 04:59:06,2.70,BKY1679,2,1.35,Renee Harmon,7A Nassau Ave,"Brooklyn, NY 11222",1999-01-14,607-231-3605,America/New_York,40.723475,-73.950965,Sesame Twist,1.02,"[6.9, 6.2, 1.6]"
91509,47077,2749,2018-05-10 04:27:58,2018-05-10 04:27:58,1.30,BKY2596,1,1.30,Renee Harmon,7A Nassau Ave,"Brooklyn, NY 11222",1999-01-14,607-231-3605,America/New_York,40.723475,-73.950965,Poppyseed Linzer Cookie,1.04,"[16.0, 7.5, 5.0]"
96542,49639,2749,2018-06-04 04:57:19,2018-06-04 04:57:19,2.22,BKY8544,2,1.11,Renee Harmon,7A Nassau Ave,"Brooklyn, NY 11222",1999-01-14,607-231-3605,America/New_York,40.723475,-73.950965,Raspberry Babka,0.88,"[15.4, 14.1, 14.0]"
229592,116028,2749,2020-03-30 04:53:08,2020-03-30 04:53:08,2.62,BKY1679,2,1.31,Renee Harmon,7A Nassau Ave,"Brooklyn, NY 11222",1999-01-14,607-231-3605,America/New_York,40.723475,-73.950965,Sesame Twist,1.02,"[6.9, 6.2, 1.6]"
276853,139378,2749,2020-11-22 04:25:45,2020-11-22 04:25:45,2.84,BKY1573,2,1.42,Renee Harmon,7A Nassau Ave,"Brooklyn, NY 11222",1999-01-14,607-231-3605,America/New_York,40.723475,-73.950965,Sesame Bagel,1.02,"[11.9, 4.7, 0.9]"
